# Safe BO：Mixed変数とq>1バッチ提案

既存の`MixedVariableSpace`、`SafetySpecification`、`SafeBOWorkflow`を組み合わせます。実行しやすいよう、安全モデルと獲得関数は解析的なデモ用クラスです。実運用では学習済みBoTorchモデルと獲得関数に置き換えます。**事後安全判定は実際の安全保証ではありません。**

In [ ]:
import torch

from robotorchan.optim.domains.variable_space import MixedVariableSpace
from robotorchan.optim.safe_workflow import SafeBOWorkflow
from robotorchan.semantics import ContinuousConstraint, SafetySpecification

torch.set_default_dtype(torch.double)
bounds = torch.tensor([[0.0, 0.0, 0.0], [1.0, 3.0, 4.0]])
space = MixedVariableSpace(
    bounds=bounds,
    integer_dims=(1,),
    categorical_values={2: (0.0, 2.0, 4.0)},
)


def safety_output(X):
    return X[..., :1] - 0.8


class DemoPosterior:
    def __init__(self, X):
        self.mean = safety_output(X)
        self.variance = torch.full_like(self.mean, 0.0001)


class DemoSafetyModel:
    def posterior(self, X):
        return DemoPosterior(X)


class DemoBatchAcquisition:
    def __call__(self, X):
        return X[..., 0].sum(dim=-1)


safety_model = DemoSafetyModel()
acquisition = DemoBatchAcquisition()

## 1. 安全な初期観測と同期バッチ提案

`ask(q=3)`は`[3, d]`の候補を返します。全点が安全と予測されたバッチだけを採用します。`tell()`には全3点の実測安全出力`[3, m]`を渡します。

In [ ]:
initial_X = torch.tensor([[0.1, 1.0, 2.0]])
specification = SafetySpecification(
    constraints=(ContinuousConstraint(0, 0.0),),
    initial_X=initial_X,
    observed_constraints=safety_output(initial_X),
    beta=4.0,
)
workflow = SafeBOWorkflow(specification, bounds, variable_space=space)

batch = workflow.ask(
    safety_model,
    mode="objective",
    acq_function=acquisition,
    q=3,
    num_samples=256,
    seed=7,
)
print("提案バッチ:", batch)
assert batch.shape == (3, 3)
assert bool((batch[:, 1] == batch[:, 1].round()).all())
assert bool(torch.isin(batch[:, 2], torch.tensor([0.0, 2.0, 4.0])).all())
assert bool((safety_output(batch) + 0.02 <= 0.0).all())

observed_constraints = safety_output(batch)
workflow.tell(batch, observed_constraints)
print("観測登録後の件数:", workflow.observed_X.shape[0])

## 2. 安全集合内の探索モード

`mode="expansion"`では安全候補の予測不確実性を評価します。ここでの探索はSafeOptの厳密な到達可能集合の計算ではありません。

In [ ]:
expansion_batch = workflow.ask(
    safety_model,
    mode="expansion",
    q=2,
    num_samples=256,
    seed=8,
)
print("探索バッチ:", expansion_batch)
assert expansion_batch.shape == (2, 3)
workflow.tell(expansion_batch, safety_output(expansion_batch))
print("観測登録後の件数:", workflow.observed_X.shape[0])

## 3. 実際のBOでの使い方

実運用では`DemoSafetyModel`を安全制約出力に学習したGPに、`DemoBatchAcquisition`をq対応の獲得関数に置き換えます。実験で得た安全制約値を`tell()`で登録した後、利用者がGPと獲得関数を更新してください。安全なバッチが見つからない場合は`RuntimeError`となり、危険候補へのフォールバックはしません。Mixed×q>1は有限バッチ候補をJoint評価する方式です。